In [1]:
pip install dlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install dlib opencv-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pygame --timeout=120

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install opencv-python dlib imutils playsound


Note: you may need to restart the kernel to use updated packages.


In [6]:
def predict_eye_state(eye_img):
    eye_img = cv2.resize(eye_img, (224, 224))  # Match MobileNetV2 input
    eye_img = cv2.cvtColor(eye_img, cv2.COLOR_GRAY2RGB)  # Convert to 3 channels
    eye_img = eye_img.astype("float32") / 255.0
    eye_img = np.expand_dims(eye_img, axis=0)  # Shape: (1, 224, 224, 3)
    pred = model.predict(eye_img)
    return int(pred[0][0] > 0.5)


In [7]:
import cv2
import dlib
import numpy as np
from scipy.spatial import distance
from keras.models import load_model
from playsound import playsound
import threading

# Load models
model = load_model('my_modell_drowsiness.keras')  # Your trained model path
predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')
detector = dlib.get_frontal_face_detector()

# Alert function
def play_alert(sound_file):
    threading.Thread(target=playsound, args=(sound_file,), daemon=True).start()

# Predict eye state (0: closed, 1: open)
def predict_eye_state(eye_img):
    # Resize the eye image to match MobileNetV2 input size (224x224)
    eye_img = cv2.resize(eye_img, (224, 224))  # Match MobileNetV2 input
    
    # Convert grayscale image to 3 channels (RGB)
    eye_img = cv2.cvtColor(eye_img, cv2.COLOR_GRAY2RGB)  # Convert to 3 channels
    
    # Normalize the pixel values to range [0, 1]
    eye_img = eye_img.astype("float32") / 255.0
    
    # Expand dimensions to match the input shape (batch size, height, width, channels)
    eye_img = np.expand_dims(eye_img, axis=0)  # Shape becomes (1, 224, 224, 3)
    
    # Predict the eye state using the trained MobileNetV2 model
    pred = model.predict(eye_img)
    
    # Return 1 (open) if the probability is greater than 0.5, else 0 (closed)
    return int(pred[0][0] > 0.5)

# Mouth Aspect Ratio
def mouth_aspect_ratio(mouth):
    A = distance.euclidean(mouth[2], mouth[10])
    B = distance.euclidean(mouth[4], mouth[8])
    C = distance.euclidean(mouth[0], mouth[6])
    return (A + B) / (2.0 * C)

# Indices
LEFT_EYE_IDX = list(range(36, 42))
RIGHT_EYE_IDX = list(range(42, 48))
MOUTH_IDX = list(range(48, 60))
MOUTH_THRESH = 0.75

# Frame count
eye_closed_frames = 0
EYE_CLOSED_LIMIT = 20

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = detector(gray)

    for face in faces:
        shape = predictor(gray, face)
        landmarks = np.array([[p.x, p.y] for p in shape.parts()])

        # Eye ROIs
        (lStart, lEnd) = (36, 42)
        (rStart, rEnd) = (42, 48)

        left_eye_pts = landmarks[lStart:lEnd]
        right_eye_pts = landmarks[rStart:rEnd]

        def crop_eye(pts):
            x, y, w, h = cv2.boundingRect(np.array(pts))
            margin = 5
            eye_img = gray[y-margin:y+h+margin, x-margin:x+w+margin]
            return eye_img

        left_eye_img = crop_eye(left_eye_pts)
        right_eye_img = crop_eye(right_eye_pts)

        # Predict using MobileNetV2 model
        left_pred = predict_eye_state(left_eye_img)
        right_pred = predict_eye_state(right_eye_img)

        # Draw eyes
        cv2.polylines(frame, [left_eye_pts], True, (255, 255, 0), 1)
        cv2.polylines(frame, [right_eye_pts], True, (255, 255, 0), 1)

        # Closed eyes check
        if left_pred == 0 and right_pred == 0:
            eye_closed_frames += 1
            if eye_closed_frames >= EYE_CLOSED_LIMIT:
                cv2.putText(frame, "DROWSINESS ALERT!", (20, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                play_alert("/Users/priyanka/Downloads/emergency-alarm-69780.wav")
        else:
            eye_closed_frames = 0

        # Yawning detection
        mouth_pts = landmarks[MOUTH_IDX]
        mar = mouth_aspect_ratio(mouth_pts)
        if mar > MOUTH_THRESH:
            cv2.putText(frame, "YAWNING ALERT!", (20, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            play_alert("/Users/priyanka/Downloads/warning-alarm-loop-1-279206.wav")

    cv2.imshow("Drowsiness and Yawn Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

/Users/priyanka/anaconda3/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 6 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━